# Colab Training Notebook

This notebook is a Colab-ready workflow for running the real Mistral + QLoRA training path for this project.

## Before you start
- In Colab, switch the runtime to GPU.
- Upload this repository to Colab, or clone it from your Git repository.
- If you use Google Drive, mount it so checkpoints and outputs persist.


## 1. Create and Configure a Google Colab Notebook

In Colab, set the runtime type to **GPU** from `Runtime -> Change runtime type`. Then run the next cell to verify the environment.


## 3. Load Data from Google Drive

Mount Drive if you want persistent storage, then point the notebook to the cloned project folder or uploaded files.


In [ ]:
import os
import sys
import platform
import subprocess

print('Python:', sys.version)
print('Platform:', platform.platform())
print('CUDA visible:', os.environ.get('CUDA_VISIBLE_DEVICES', 'not set'))
try:
    import torch
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU name:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch check failed:', exc)


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary_path = RESULTS_DIR / 'colab_dataset_summary.json'
with summary_path.open('w', encoding='utf-8') as f:
    json.dump(
        {
            'dataset_rows': len(df),
            'instruction_len_mean': float(df['instruction_len'].mean()),
            'output_len_mean': float(df['output_len'].mean()),
        },
        f,
        indent=2,
    )
print('Saved summary to:', summary_path)

print('Notebook is ready for the next step: running src.training.train on a Colab GPU runtime.')


## 6. Save Outputs and Export Notebook

Save generated artifacts to Drive so they persist after the Colab session ends.


In [ ]:
df = pd.DataFrame(rows)
df['instruction_len'] = df['instruction'].str.len()
df['output_len'] = df['output'].str.len()

plt.figure(figsize=(10, 4))
sns.histplot(df['output_len'], bins=20, color='steelblue')
plt.title('Output Length Distribution')
plt.xlabel('Characters')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 5. Visualize Results

Use a small chart to inspect dataset length distribution before training. This is a quick sanity check for content variety.


In [ ]:
import random

start = time.time()
with DATA_PATH.open('r', encoding='utf-8') as f:
    rows = json.load(f)

sample = random.sample(rows, k=min(5, len(rows)))
print('Sample count:', len(sample))
print('Sample instruction length:', len(sample[0]['instruction']) if sample else 0)
print('Sample output length:', len(sample[0]['output']) if sample else 0)
print('Elapsed seconds:', round(time.time() - start, 3))


In [ ]:
BASE_OUTPUT = RESULTS_DIR / 'base_outputs' / 'predictions.json'
FINETUNED_OUTPUT = RESULTS_DIR / 'finetuned_outputs' / 'predictions.json'
METRICS_OUTPUT = RESULTS_DIR / 'metrics' / 'comparison_report.json'

# Mock-free local commands. Adjust the model path if you save checkpoints somewhere else.
!python -m src.evaluation.inference --output {BASE_OUTPUT} --max-samples 25
!python -m src.evaluation.inference --model-name {RESULTS_DIR / 'finetuned_model'} --output {FINETUNED_OUTPUT} --max-samples 25
!python -m src.evaluation.compare --base-file {BASE_OUTPUT} --finetuned-file {FINETUNED_OUTPUT} --report-file {METRICS_OUTPUT}


## 4c. Generate Baseline and Fine-Tuned Outputs

After training finishes, generate comparison outputs for the base model and the fine-tuned checkpoint.


In [ ]:
import os

os.environ['USE_4BIT'] = 'true'
os.environ['MODEL_NAME'] = 'mistralai/Mistral-7B-Instruct-v0.2'
os.environ['MAX_LENGTH'] = '1024'

print('USE_4BIT =', os.environ['USE_4BIT'])
print('MODEL_NAME =', os.environ['MODEL_NAME'])
print('MAX_LENGTH =', os.environ['MAX_LENGTH'])

# From the repo root in Colab, run the training pipeline.
# If you want a faster proof-of-life test first, reduce the sample counts.
!python -m src.training.train --max-train-samples 128 --max-eval-samples 32


## 4b. Run Real Mistral + QLoRA Training

Set the project to use 4-bit loading in Colab, then run the training entrypoint from the repository root.


## 4. Run a Baseline Computation

Before long training, run a quick sanity check on the dataset and the code path. This confirms the notebook is wired correctly.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Update this to match where you place the repository in Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/fine-tune-llm')
DATA_PATH = PROJECT_ROOT / 'data' / 'final_dataset.json'
RESULTS_DIR = PROJECT_ROOT / 'results'

print('Project root:', PROJECT_ROOT)
print('Dataset exists:', DATA_PATH.exists())
if DATA_PATH.exists():
    with DATA_PATH.open('r', encoding='utf-8') as f:
        data = json.load(f)
    print('Dataset rows:', len(data))
    print('First row keys:', list(data[0].keys()) if data else [])


In [ ]:
!pip -q install transformers==4.46.3 datasets==2.19.0 peft==0.18.1 trl==0.8.6 bitsandbytes accelerate==1.13.0 sentencepiece pandas matplotlib seaborn

import json
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Install and Import Dependencies

Install the project dependencies in Colab. Keep the versions pinned so the notebook matches the local code path.
